# ADS v2: Home renewal events

Builds 46 event families: `HR0`, `HR0.F1`, `HR0.F2`, `HR0.F3`, `HR1a`, `HR1b`, `HR1.E1`, `HR1.E1.F1`, `HR2a`, `HR2b`, `HR2.F1`, `HR3b.F1`, `HR3a`, `HR4a`, `HR4c`, `HR3b`, `HR3.O1`, `HR4`, `HR3.B1.1`, `HR2.B1.1`, `HR2.B1.F1`, `HR2.B1.2`, `HR2.B1.2.F1`, `HR2.B1.3.F1`, `HR2.B1.O1`, `HR2.B1.O2`, `HR2.B1.O3`, `HR3.B1.1.F1`, `HR3.B1.2`, `HR3.B1.2.F1`, `HR3.B1.O1`, `HR3.B1.O3`, `HR5`, `HR6`, `HR6.F1`, `HR4.B2.F1`, `HR4.B2.4`, `HR4.B2.1`, `HR4.B2.2`, `HR4.B2.5`, `HR4.B2.3`, `HR4.B2.6`, `HR4.B2.O1`, `HR6.B1.1`, `HR6.B1.2`, `HR6.B1.5`.

Generated by `ads_build/v2/build_ads.py` from `families/*.yaml`. Event descriptions come from `seeds/ChillEpisodeTypes.csv`.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ads_build/v2/families/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run ADS_v2_00_Setup

In [ ]:
%%pyspark
ads_run_log("14_Home_Renewal", "started")


## HR0: Renewal window opens at T-40


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr0  (HR0)                                                              */
/*  Description : Renewals / Spine. Notebook 04 cell 196. The renewal window opens 40 days before the r */
/*              : enew date, per renewing home policy and month. Change: events outside the build windo */
/*              : w are dropped (04 had no window).                                                     */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyRenewal                                 */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr0 as
select  xxhash64('HR0', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR0'                                               as EventTypeId,
        'Renewal window opens at T-40' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(min(date_sub(RenewalDate, 40)) as timestamp)               as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Home'
        and     cast(date_sub(RenewalDate, 40) as timestamp) >= timestamp '${ads.start_time}' and cast(date_sub(RenewalDate, 40) as timestamp) < timestamp '${ads.end_time}'
        group by PolicyCode, date_trunc('MONTH', RenewalDate)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR0.F1: Declared claims not recorded


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_home_renewal_claim_events                                                   */
/*  Description : Home QS transactions on claim risks of renewing home policies (04 cells 197-199).     */
/*              :                                                                                       */
/*              : Reads: ods_v2.PolicyHomeQsEvent, ods_v2.PolicyClaimHome, stg.adsv2_renewal_base       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_home_renewal_claim_events as
select  e.PolicyCode, e.EventType, e.TransactionDate
from    ${ads.ods}.PolicyHomeQsEvent e
left semi join ${ads.ods}.PolicyClaimHome c on c.HomeRiskId = e.HomeRiskId
join    (select distinct PolicyCode from ${ads.stg}.adsv2_renewal_base where LYPolicyTypeGroup = 'Home') b
        on b.PolicyCode = e.PolicyCode
where   cast(e.TransactionDate as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(e.TransactionDate as timestamp) < timestamp '${ads.end_time}'

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr0_f1  (HR0.F1)                                                        */
/*  Description : Renewals / Friction. Notebook 04 cell 197. Claim risks with a 'Quotation Provided' tr */
/*              : ansaction (declared claims not recorded).                                             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyClaimHome, PolicyHomeQsEvent, PolicyRen */
/*              : ewal                                                                                  */
/*              : Reads: stg.adsv2_home_renewal_claim_events                                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr0_f1 as
select  xxhash64('HR0.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR0.F1'                                            as EventTypeId,
        'Declared claims not recorded' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(min(TransactionDate) as timestamp)                         as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_home_renewal_claim_events
        where   EventType = 'Quotation Provided'
        group by PolicyCode, date_trunc('MONTH', TransactionDate)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR0.F2: Home Renewal - Renewal held letter issued


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr0_f2  (HR0.F2)                                                        */
/*  Description : Renewals / Friction. Notebook 04 cell 198. Claim risks with a 'Policy Renewal Accepte */
/*              : d' transaction (renewal held letter issued).                                          */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyClaimHome, PolicyHomeQsEvent, PolicyRen */
/*              : ewal                                                                                  */
/*              : Reads: stg.adsv2_home_renewal_claim_events                                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr0_f2 as
select  xxhash64('HR0.F2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR0.F2'                                            as EventTypeId,
        'Home Renewal - Renewal held letter issued' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(min(TransactionDate) as timestamp)                         as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_home_renewal_claim_events
        where   EventType = 'Policy Renewal Accepted'
        group by PolicyCode, date_trunc('MONTH', TransactionDate)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR0.F3: Home Renewal - No terms from the insurer, so no offer is made


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr0_f3  (HR0.F3)                                                        */
/*  Description : Renewals / Friction. Notebook 04 cell 199. Claim risks with a 'Policy Lapsed' transac */
/*              : tion (no terms from the insurer, no offer).                                           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyClaimHome, PolicyHomeQsEvent, PolicyRen */
/*              : ewal                                                                                  */
/*              : Reads: stg.adsv2_home_renewal_claim_events                                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr0_f3 as
select  xxhash64('HR0.F3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR0.F3'                                            as EventTypeId,
        'Home Renewal - No terms from the insurer, so no offer is made' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(min(TransactionDate) as timestamp)                         as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_home_renewal_claim_events
        where   EventType = 'Policy Lapsed'
        group by PolicyCode, date_trunc('MONTH', TransactionDate)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR1a: Renewal invitation sent by EMAIL


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr1a  (HR1a)                                                            */
/*  Description : Renewals / Spine. Notebook 04 cells 41, 200. The first 'Renewal Offer - Emailed Docum */
/*              : ent' per renewing home policy.                                                        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyDocumentEvent, PolicyRenewal            */
/*              : Reads: stg.adsv2_relay_document_event, stg.adsv2_renewal_base                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr1a as
select  xxhash64('HR1a', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR1a'                                              as EventTypeId,
        'Renewal invitation sent by EMAIL' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                d.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_relay_document_event d
        join    (select distinct PolicyCode from ${ads.stg}.adsv2_renewal_base where LYPolicyTypeGroup = 'Home') b
                on b.PolicyCode = d.PolicyCode
        where   d.EventDescription like 'Renewal Offer - Emailed Document %'
        and     d.PolicyTypeGroup = 'Home'
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        group by d.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR1b: Renewal invitation sent by POST


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr1b  (HR1b)                                                            */
/*  Description : Renewals / Spine. Notebook 04 cells 42, 201. The first 'Renewal Offer - Document Tran */
/*              : smitted' (posted) per renewing home policy.                                           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyDocumentEvent, PolicyRenewal            */
/*              : Reads: stg.adsv2_relay_document_event, stg.adsv2_renewal_base                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr1b as
select  xxhash64('HR1b', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR1b'                                              as EventTypeId,
        'Renewal invitation sent by POST' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                d.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_relay_document_event d
        join    (select distinct PolicyCode from ${ads.stg}.adsv2_renewal_base where LYPolicyTypeGroup = 'Home') b
                on b.PolicyCode = d.PolicyCode
        where   d.EventDescription like 'Renewal Offer - Document Transmitted %'
        and     d.PolicyTypeGroup = 'Home'
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        group by d.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR1.E1: Renewal campaigns: CRM


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr1_e1  (HR1.E1)                                                        */
/*  Description : Renewals / Chill Effort. Home renewal CRM campaigns (email / SMS sent) to renewing ho */
/*              : me policies, first per policy and month. Change: 04 never wrote HR1.E1 (cell 206 wrot */
/*              : e HR1.E1.F1, a copy of cell 207: design bug list); this is the rule the Motor R1.E1 w */
/*              : as mirrored from, on the Xtremepush events.                                           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: MessageEvent, PolicyRenewal                   */
/*              : Reads: ods_v2.MessageEvent, stg.adsv2_renewal_base                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr1_e1 as
select  xxhash64('HR1.E1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR1.E1'                                            as EventTypeId,
        'Renewal campaigns: CRM' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(m.EventDateTime)                                            as EventDateTime,
                21                                                              as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.MessageEvent m
        join    (select distinct PolicyCode from ${ads.stg}.adsv2_renewal_base where LYPolicyTypeGroup = 'Home') b
                on b.PolicyCode = m.PolicyCode
        where   m.CampaignName like '%Home%'
        and     m.CampaignName like '%Renewals%'
        and     m.MessageType in ('EMAIL', 'SMS')
        and     m.InteractionType = 'sent'
        and     m.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and m.EventDateTime < add_months(timestamp '${ads.end_time}', 1)
        group by m.PolicyCode, date_trunc('MONTH', m.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR1.E1.F1: Programme fails and the renewal date is missed


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr1_e1_f1  (HR1.E1.F1)                                                  */
/*  Description : Renewals / Friction. Notebook 04 cell 207. Home renewal campaigns sent (policy feed)  */
/*              : to policies that renewed only after the renew date (the programme missed the date), a */
/*              : t the last send per policy and recipient. Written once (04 wrote it twice, cells 206  */
/*              : and 207).                                                                             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: MessageEvent, PolicyRenewal                   */
/*              : Reads: ods_v2.MessageEvent, stg.adsv2_renewal_base                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr1_e1_f1 as
select  xxhash64('HR1.E1.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR1.E1.F1'                                         as EventTypeId,
        'Programme fails and the renewal date is missed' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(m.EventDateTime)                                            as EventDateTime,
                21                                                              as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                m.EmailKey                                                      as SourceCustomerReference,
                case when m.EmailKey is not null then 'EmailKey' end            as SourceCustomerReferenceType
        from    ${ads.ods}.MessageEvent m
        join    ${ads.stg}.adsv2_renewal_base b on b.PolicyCode = m.PolicyCode and b.LYPolicyTypeGroup = 'Home'
        where   m.SourceFeed = 'Policy'
        and     m.CampaignName like '%Home%'
        and     m.CampaignName like '%Renewals%'
        and     m.MessageType in ('EMAIL', 'SMS')
        and     m.InteractionType = 'sent'
        and     m.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and m.EventDateTime < add_months(timestamp '${ads.end_time}', 1)
        and     coalesce(b.TYReportingSaleDate, b.RenewalDate) > b.RenewalDate
        group by m.PolicyCode, m.EmailKey
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR2a: Home Renewal - Customer logs in to portal


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr2a  (HR2a)                                                            */
/*  Description : Renewals / Spine. Notebook 04 cell 202. Portal logins by the client from 40 days befo */
/*              : re to 40 days after the renew date, first per policy and month.                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: RenewalPortalEvent, PolicyRenewal             */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.RenewalPortalEvent                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr2a as
select  xxhash64('HR2a', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR2a'                                              as EventTypeId,
        'Home Renewal - Customer logs in to portal' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(e.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.RenewalPortalEvent e on e.PortfolioCode = left(b.PolicyCode, 6)
        where   b.LYPolicyTypeGroup = 'Home'
        and     e.EventType = 'LoginSuccess'
        and     e.EventDateTime > date_sub(b.RenewalDate, 40)
        and     e.EventDateTime < date_add(b.RenewalDate, 40)
        group by b.PolicyCode, date_trunc('MONTH', e.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR2b: Home Renewal - Customer phones call center


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr2b  (HR2b)                                                            */
/*  Description : Renewals / Spine. Notebook 04 cell 203. Conversations on the INBOUND_Home_Renewals qu */
/*              : eue, at their start. Change: source system 20 Genesys.                                */
/*              :                                                                                       */
/*              : Grain: Call   Product: Home   Source: CallSession                                     */
/*              : Reads: ods_v2.CallSession                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr2b as
select  xxhash64('HR2b', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR2b'                                              as EventTypeId,
        'Home Renewal - Customer phones call center' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(ConversationStartTime)                                      as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference
        from    ${ads.ods}.CallSession
        where   QueueName = 'INBOUND_Home_Renewals'
        and     ConversationStartTime >= timestamp '${ads.start_time}' and ConversationStartTime < timestamp '${ads.end_time}'
        group by ConversationId
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR2.F1: Customer does not accept the price


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr2_f1  (HR2.F1)                                                        */
/*  Description : Renewals / Friction. Notebook 04 cell 204. Inbound home renewal calls wrapped up as u */
/*              : nfavourable NCD, re-quoted not sold, undecided / shopping around or unfavourable comp */
/*              : etitor (customer does not accept the price).                                          */
/*              :                                                                                       */
/*              : Grain: Call   Product: Home   Source: CallSession                                     */
/*              : Reads: ods_v2.CallSession                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr2_f1 as
select  xxhash64('HR2.F1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR2.F1'                                            as EventTypeId,
        'Customer does not accept the price' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(ConversationStartTime)                                      as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference
        from    ${ads.ods}.CallSession
        where   WrapupCode <> 'ININ-WRAP-UP-TIMEOUT'
        and     QueueName = 'INBOUND_Home_Renewals'
        and     OriginatingDirection = 'inbound'
        and     WrapupCodeName in ('UNFAVOURABLE_NCD', 'REQUOTED_NOT_SOLD', 'UNDECIDED/SHOPPING AROUND', 'UNFAVOURABLE_COMPETITOR')
        and     ConversationStartTime >= add_months(timestamp '${ads.start_time}', -1) and ConversationStartTime < timestamp '${ads.end_time}'
        group by ConversationId
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR3b.F1: Call wait time-  or fails to make contact


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr3b_f1  (HR3b.F1)                                                      */
/*  Description : Renewals / Friction. Notebook 04 cell 210. Inbound home renewal calls that waited mor */
/*              : e than 600 seconds or were abandoned (D-E1.25; 04 used the NO_ANSWER wrap-up code ins */
/*              : tead).                                                                                */
/*              :                                                                                       */
/*              : Grain: Call   Product: Home   Source: CallSession                                     */
/*              : Reads: ods_v2.CallSession                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr3b_f1 as
select  xxhash64('HR3b.F1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR3b.F1'                                           as EventTypeId,
        'Call wait time-  or fails to make contact' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(ConversationStartTime)                                      as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference
        from    ${ads.ods}.CallSession
        where   QueueName = 'INBOUND_Home_Renewals'
        and     ConversationStartTime >= timestamp '${ads.start_time}' and ConversationStartTime < timestamp '${ads.end_time}'
        group by ConversationId
        having  sum(AcdWaitDurationMs) > 600000 or sum(Abandoned) > 0
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR3a: Customer renews ONLINE


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr3a  (HR3a)                                                            */
/*  Description : Renewals / Spine. Notebook 04 cell 208. A portal payment for the policy itself from 6 */
/*              : 0 days before to 40 days after the renew date.                                        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: RenewalPortalEvent, PolicyRenewal             */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.RenewalPortalEvent                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr3a as
select  xxhash64('HR3a', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR3a'                                              as EventTypeId,
        'Customer renews ONLINE' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(e.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.RenewalPortalEvent e on e.PolicyCode = b.PolicyCode
        where   b.LYPolicyTypeGroup = 'Home'
        and     e.EventType = 'PaymentSuccess'
        and     e.EventDateTime > date_sub(b.RenewalDate, 60)
        and     e.EventDateTime < date_add(b.RenewalDate, 40)
        group by b.PolicyCode, date_trunc('MONTH', e.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR4a: Pay in full


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_home_renewal_online_payment                                                 */
/*  Description : The last portal payment per renewing home policy's client in its renewal period (stg. */
/*              : HR4_HomeRenewalsOnlinePayments).                                                      */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.RenewalPortalEvent                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_home_renewal_online_payment as
select  left(b.PolicyCode, 6) as ClientCode, b.PolicyCode, max(e.EventDateTime) as PaymentDateTime
from    ${ads.stg}.adsv2_renewal_base b
join    ${ads.ods}.RenewalPortalEvent e on e.PortfolioCode = left(b.PolicyCode, 6)
where   b.LYPolicyTypeGroup = 'Home'
and     e.EventType = 'PaymentSuccess'
and     e.EventDateTime > date_sub(b.RenewalDate, 60)
and     e.EventDateTime < date_add(b.RenewalDate, 40)
group by left(b.PolicyCode, 6), b.PolicyCode

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr4a  (HR4a)                                                            */
/*  Description : Renewals / Spine. Notebook 04 cells 46, 214. The client's last portal payment in the  */
/*              : renewal period (60 days before to 40 after). As 04, no pay-in-full filter is applied  */
/*              : (04's query has none).                                                                */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: RenewalPortalEvent, PolicyRenewal             */
/*              : Reads: stg.adsv2_home_renewal_online_payment                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr4a as
select  xxhash64('HR4a', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR4a'                                              as EventTypeId,
        'Pay in full' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(PaymentDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_home_renewal_online_payment
        group by PolicyCode, date_trunc('MONTH', PaymentDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR4c: New monthly agreement


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr4c  (HR4c)                                                            */
/*  Description : Renewals / Spine. Notebook 04 cell 215. As HR4a where that last payment is by instalm */
/*              : ents (a new monthly agreement).                                                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: RenewalPortalEvent, PolicyRenewal             */
/*              : Reads: stg.adsv2_home_renewal_online_payment, ods_v2.RenewalPortalEvent               */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr4c as
select  xxhash64('HR4c', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR4c'                                              as EventTypeId,
        'New monthly agreement' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(p.PaymentDateTime)                                          as EventDateTime,
                1                                                               as SourceSystemId,
                p.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_home_renewal_online_payment p
        join    ${ads.ods}.RenewalPortalEvent e on e.PortfolioCode = p.ClientCode and e.EventDateTime = p.PaymentDateTime
        where   e.EventType = 'PaymentSuccess'
        and     e.PaymentType = 'Instalments'
        group by p.PolicyCode, date_trunc('MONTH', p.PaymentDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR3b: Customer renews on an INBOUND call


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr3b  (HR3b)                                                            */
/*  Description : Renewals / Spine. Notebook 04 cell 209. Retained home renewals on channel 'a) IB', on */
/*              :  the new policy at its sale date.                                                     */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyRenewal                                 */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr3b as
select  xxhash64('HR3b', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR3b'                                              as EventTypeId,
        'Customer renews on an INBOUND call' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(TYReportingSaleDate) as timestamp)                     as EventDateTime,
                1                                                               as SourceSystemId,
                TYPolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Home'
        and     PolicyRetNum = 1
        and     Channel = 'a) IB'
        group by TYPolicyCode, date_trunc('MONTH', TYReportingSaleDate)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR3.O1: Policy lapsed


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr3_o1  (HR3.O1)                                                        */
/*  Description : Renewals / Outcome. Notebook 04, corrected (H10, D-E1.21): the policy lapsed in this  */
/*              : cycle = lost (not renewed within the window), from the cycle's own result, at due + 3 */
/*              : 0 days. 04 wrote the policy's current status L at every renewal date.                 */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyRenewal, PolicySale, RenewalPortalEvent */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr3_o1 as
select  xxhash64('HR3.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR3.O1'                                            as EventTypeId,
        'Policy lapsed' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(date_add(RenewalDate, 30) as timestamp)                       as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Home'
        and     Lost = 1
        and     OutcomeFinal = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR4: Payment taken


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr4  (HR4)                                                              */
/*  Description : Renewals / Spine. Notebook 04 cell 212. Retained home renewals (payment taken), on th */
/*              : e new policy at its sale date.                                                        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyRenewal                                 */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr4 as
select  xxhash64('HR4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR4'                                               as EventTypeId,
        'Payment taken' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(TYReportingSaleDate) as timestamp)                     as EventDateTime,
                1                                                               as SourceSystemId,
                TYPolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Home'
        and     PolicyRetNum = 1
        group by TYPolicyCode, date_trunc('MONTH', TYReportingSaleDate)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR3.B1.1: Renewal date passes with no renewal


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr3_b1_1  (HR3.B1.1)                                                    */
/*  Description : Renewals / Branch. Notebook 04, corrected. Lapsed = no renewal by the due date (D-E1. */
/*              : 21), from the cycle's resolved renewal (finance retained, else the +/-30-day match; D */
/*              : -E1.19). One event per cycle, at the due date, once the due date has passed. 04 count */
/*              : ed renewals not yet due as lapsed, and (Home) left out lapses with no renewal at all. */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyRenewal, PolicySale, RenewalPortalEvent */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr3_b1_1 as
select  xxhash64('HR3.B1.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR3.B1.1'                                          as EventTypeId,
        'Renewal date passes with no renewal' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(RenewalDate as timestamp)                                     as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Home'
        and     Lapsed = 1
        and     RenewalDate < date '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR2.B1.1: Re-quote started from the renewal price


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_home_renewal_hfq                                                            */
/*  Description : HFQ re-quotes of each renewing home cycle, due - 60 to due + 30 days (H8), with the o */
/*              : ffer and how the cycle renewed.                                                       */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_renewal_scv_quotes, stg.adsv2_renewal_base                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_home_renewal_hfq as
select  distinct q.PolicyCode, q.RenewalDate, q.QuoteReference as QuoteCodeReference, q.QuoteId, q.QuoteDateTime,
        b.RenEURPremOffer, b.TYEURGrossPremium, b.PolicyRetNum, b.Renewed, b.RenewedSameCode, b.OutcomeFinal
from    ${ads.stg}.adsv2_renewal_scv_quotes q
join    ${ads.stg}.adsv2_renewal_base b on b.PolicyCode = q.PolicyCode and b.RenewalDate = q.RenewalDate
where   q.LYPolicyTypeGroup = 'Home'
and     q.QuoteSourceSystemId = 3

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr2_b1_1  (HR2.B1.1)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cells 43, 44, 47, 219, corrected (H8, D-E1.27): an HFQ */
/*              :  re-quote of the cycle (same customer, due - 60 to due + 30), one event per re-quote. */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: Quote, PolicyRenewal, scveadmlinkfinal        */
/*              : Reads: stg.adsv2_home_renewal_hfq                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr2_b1_1 as
select  xxhash64('HR2.B1.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR2.B1.1'                                          as EventTypeId,
        'Re-quote started from the renewal price' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(QuoteDateTime)                                              as EventDateTime,
                3                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                QuoteCodeReference                                              as SourceQuoteReference,
                max(QuoteId)                                                    as QuoteId
        from    ${ads.stg}.adsv2_home_renewal_hfq
        group by PolicyCode, RenewalDate, QuoteCodeReference
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR2.B1.F1: The customer shops the renewal at all


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr2_b1_f1  (HR2.B1.F1)                                                  */
/*  Description : Renewals / Friction. Notebook 04 cell 205. A successful HFQ payment on a re-quote of  */
/*              : the cycle (shops the renewal), at the first such payment.                             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: Quote, QuoteHomePaymentResponse, scveadmlinkf */
/*              : inal                                                                                  */
/*              : Reads: stg.adsv2_home_renewal_hfq, stg.adsv2_hfq_payment                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr2_b1_f1 as
select  xxhash64('HR2.B1.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR2.B1.F1'                                         as EventTypeId,
        'The customer shops the renewal at all' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(p.ActionTimeCreated)                                        as EventDateTime,
                3                                                               as SourceSystemId,
                h.PolicyCode                                                    as SourcePolicyReference,
                h.QuoteCodeReference                                            as SourceQuoteReference,
                max(h.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_home_renewal_hfq h
        join    ${ads.stg}.adsv2_hfq_payment p  on p.QuoteReference = h.QuoteCodeReference
        where   p.ActionResultCode = 'SUCCESS'
        and     p.ActionTimeCreated >= timestamp '${ads.start_time}' and p.ActionTimeCreated < timestamp '${ads.end_time}'
        group by h.PolicyCode, h.RenewalDate, h.QuoteCodeReference
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR2.B1.2: New price returned


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr2_b1_2  (HR2.B1.2)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cells 15, 220. A re-quote of the cycle with a premium  */
/*              : returned (new price returned), one event per re-quote.                                */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: Quote, QuoteHomePrice, PolicyRenewal, scveadm */
/*              : linkfinal                                                                             */
/*              : Reads: stg.adsv2_home_renewal_hfq, stg.adsv2_hfq_priced                               */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr2_b1_2 as
select  xxhash64('HR2.B1.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR2.B1.2'                                          as EventTypeId,
        'New price returned' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(h.QuoteDateTime)                                            as EventDateTime,
                3                                                               as SourceSystemId,
                h.PolicyCode                                                    as SourcePolicyReference,
                h.QuoteCodeReference                                            as SourceQuoteReference,
                max(h.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_home_renewal_hfq h
        left semi join ${ads.stg}.adsv2_hfq_priced p on p.QuoteCodeReference = h.QuoteCodeReference
        group by h.PolicyCode, h.RenewalDate, h.QuoteCodeReference
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR2.B1.2.F1: No price returned-  or the quote is abandoned


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr2_b1_2_f1  (HR2.B1.2.F1)                                              */
/*  Description : Renewals / Friction. Notebook 04 cell 221, corrected (H8): re-quotes of the cycle wit */
/*              : h no premium returned, one event per re-quote.                                        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: Quote, QuoteHomePrice, PolicyRenewal, scveadm */
/*              : linkfinal                                                                             */
/*              : Reads: stg.adsv2_home_renewal_hfq, stg.adsv2_hfq_priced                               */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr2_b1_2_f1 as
select  xxhash64('HR2.B1.2.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR2.B1.2.F1'                                       as EventTypeId,
        'No price returned-  or the quote is abandoned' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(h.QuoteDateTime)                                            as EventDateTime,
                3                                                               as SourceSystemId,
                h.PolicyCode                                                    as SourcePolicyReference,
                h.QuoteCodeReference                                            as SourceQuoteReference,
                max(h.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_home_renewal_hfq h
        left anti join (select distinct QuoteCodeReference from ${ads.stg}.adsv2_hfq_priced) p
                on p.QuoteCodeReference = h.QuoteCodeReference
        group by h.PolicyCode, h.RenewalDate, h.QuoteCodeReference
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR2.B1.3.F1: New price is worse than the renewal offer


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr2_b1_3_f1  (HR2.B1.3.F1)                                              */
/*  Description : Renewals / Friction. Notebook 04 cell 222, corrected (H8, D-E1.27): the re-quote's be */
/*              : st price (lowest total payable returned) is worse than the renewal offer. 04 compared */
/*              :  it with the premium finally paid.                                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: Quote, QuoteHomePrice, PolicyRenewal          */
/*              : Reads: stg.adsv2_home_renewal_hfq, stg.adsv2_hfq_priced                               */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr2_b1_3_f1 as
select  xxhash64('HR2.B1.3.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR2.B1.3.F1'                                       as EventTypeId,
        'New price is worse than the renewal offer' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(h.QuoteDateTime)                                            as EventDateTime,
                3                                                               as SourceSystemId,
                h.PolicyCode                                                    as SourcePolicyReference,
                h.QuoteCodeReference                                            as SourceQuoteReference,
                max(h.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_home_renewal_hfq h
        join    (select QuoteCodeReference, min(TotalPayable) as QuoteBestPrice from ${ads.stg}.adsv2_hfq_priced group by QuoteCodeReference) p
                on p.QuoteCodeReference = h.QuoteCodeReference
        where   p.QuoteBestPrice > h.RenEURPremOffer
        and     h.RenEURPremOffer > 0
        group by h.PolicyCode, h.RenewalDate, h.QuoteCodeReference
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR2.B1.O1: Renews on the renewal offer


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr2_b1_o1  (HR2.B1.O1)                                                  */
/*  Description : Renewals / Outcome. Notebook 04 cell 223, corrected (H8, D-E1.27): a priced re-quote, */
/*              :  and the cycle renewed on the renewal offer (same policy code).                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: Quote, QuoteHomePrice, PolicyRenewal, scveadm */
/*              : linkfinal                                                                             */
/*              : Reads: stg.adsv2_home_renewal_hfq, stg.adsv2_hfq_priced                               */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr2_b1_o1 as
select  xxhash64('HR2.B1.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR2.B1.O1'                                         as EventTypeId,
        'Renews on the renewal offer' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(h.QuoteDateTime)                                            as EventDateTime,
                3                                                               as SourceSystemId,
                h.PolicyCode                                                    as SourcePolicyReference,
                h.QuoteCodeReference                                            as SourceQuoteReference,
                max(h.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_home_renewal_hfq h
        left semi join ${ads.stg}.adsv2_hfq_priced p on p.QuoteCodeReference = h.QuoteCodeReference
        where   h.Renewed = 1
        and     h.RenewedSameCode = 1
        group by h.PolicyCode, h.RenewalDate, h.QuoteCodeReference
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR2.B1.O2: Renews on a new price


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr2_b1_o2  (HR2.B1.O2)                                                  */
/*  Description : Renewals / Outcome. Notebook 04 cell 224, corrected (H8, D-E1.27): a priced re-quote, */
/*              :  and the cycle renewed on a new price (a new policy code).                            */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: Quote, QuoteHomePrice, PolicyRenewal, scveadm */
/*              : linkfinal                                                                             */
/*              : Reads: stg.adsv2_home_renewal_hfq, stg.adsv2_hfq_priced                               */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr2_b1_o2 as
select  xxhash64('HR2.B1.O2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR2.B1.O2'                                         as EventTypeId,
        'Renews on a new price' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(h.QuoteDateTime)                                            as EventDateTime,
                3                                                               as SourceSystemId,
                h.PolicyCode                                                    as SourcePolicyReference,
                h.QuoteCodeReference                                            as SourceQuoteReference,
                max(h.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_home_renewal_hfq h
        left semi join ${ads.stg}.adsv2_hfq_priced p on p.QuoteCodeReference = h.QuoteCodeReference
        where   h.Renewed = 1
        and     h.RenewedSameCode = 0
        group by h.PolicyCode, h.RenewalDate, h.QuoteCodeReference
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR2.B1.O3: Buys elsewhere-  no renewal


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr2_b1_o3  (HR2.B1.O3)                                                  */
/*  Description : Renewals / Outcome. Notebook 04 cell 225, corrected (H8, D-E1.27): a re-quote, and th */
/*              : e cycle did not renew (buys elsewhere), once the outcome is final.                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: Quote, QuoteHomePrice, PolicyRenewal, scveadm */
/*              : linkfinal                                                                             */
/*              : Reads: stg.adsv2_home_renewal_hfq                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr2_b1_o3 as
select  xxhash64('HR2.B1.O3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR2.B1.O3'                                         as EventTypeId,
        'Buys elsewhere-  no renewal' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(h.QuoteDateTime)                                            as EventDateTime,
                3                                                               as SourceSystemId,
                h.PolicyCode                                                    as SourcePolicyReference,
                max(h.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_home_renewal_hfq h
        where   h.Renewed = 0
        and     h.OutcomeFinal = 1
        group by h.PolicyCode, h.RenewalDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR3.B1.1.F1: Home Renewal - CRM bounces


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr3_b1_1_f1  (HR3.B1.1.F1)                                              */
/*  Description : Renewals / Friction. Notebook 04 cell 227. Bounced home 'Lapsed' campaign messages to */
/*              :  renewing home policies, first per policy and month. Change: the home renewal base (0 */
/*              : 4 used the Motor base here: design bug list).                                         */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: MessageEvent, PolicyRenewal                   */
/*              : Reads: ods_v2.MessageEvent, stg.adsv2_renewal_base                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr3_b1_1_f1 as
select  xxhash64('HR3.B1.1.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR3.B1.1.F1'                                       as EventTypeId,
        'Home Renewal - CRM bounces' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(m.EventDateTime)                                            as EventDateTime,
                21                                                              as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.MessageEvent m
        join    (select distinct PolicyCode from ${ads.stg}.adsv2_renewal_base where LYPolicyTypeGroup = 'Home') b
                on b.PolicyCode = m.PolicyCode
        where   m.CampaignName like '%Home%'
        and     m.CampaignName like '%Lapsed%'
        and     m.MessageType in ('EMAIL', 'SMS')
        and     m.InteractionType = 'bounce'
        and     m.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and m.EventDateTime < timestamp '${ads.end_time}'
        group by m.PolicyCode, date_trunc('MONTH', m.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR3.B1.2: Lapse CRM sequence runs


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr3_b1_2  (HR3.B1.2)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 228. Home 'Lapsed' campaign messages sent to rene */
/*              : wing home policies (the lapse sequence runs).                                         */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: MessageEvent, PolicyRenewal                   */
/*              : Reads: ods_v2.MessageEvent, stg.adsv2_renewal_base                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr3_b1_2 as
select  xxhash64('HR3.B1.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR3.B1.2'                                          as EventTypeId,
        'Lapse CRM sequence runs' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(m.EventDateTime)                                            as EventDateTime,
                21                                                              as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.MessageEvent m
        join    (select distinct PolicyCode from ${ads.stg}.adsv2_renewal_base where LYPolicyTypeGroup = 'Home') b
                on b.PolicyCode = m.PolicyCode
        where   m.CampaignName like '%Home%'
        and     m.CampaignName like '%Lapsed%'
        and     m.MessageType in ('EMAIL', 'SMS')
        and     m.InteractionType = 'sent'
        and     m.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and m.EventDateTime < timestamp '${ads.end_time}'
        group by m.PolicyCode, date_trunc('MONTH', m.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR3.B1.2.F1: Unreachable on the lapse sequence


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr3_b1_2_f1  (HR3.B1.2.F1)                                              */
/*  Description : Renewals / Friction. Notebook 04 cell 229, on the rule Vijay set for the Motor twin ( */
/*              : 2026-10-19): renewing home policies sent a home 'Lapsed' campaign email that was neve */
/*              : r opened (no open or click) or that bounced; one per policy and month, at the last se */
/*              : nd. 04 required both no open and a bounce.                                            */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: MessageEvent, PolicyRenewal                   */
/*              : Reads: ods_v2.MessageEvent, stg.adsv2_renewal_base                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr3_b1_2_f1 as
select  xxhash64('HR3.B1.2.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR3.B1.2.F1'                                       as EventTypeId,
        'Unreachable on the lapse sequence' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(case when m.InteractionType = 'sent' then m.EventDateTime end) as EventDateTime,
                21                                                              as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.MessageEvent m
        join    (select distinct PolicyCode from ${ads.stg}.adsv2_renewal_base where LYPolicyTypeGroup = 'Home') b
                on b.PolicyCode = m.PolicyCode
        where   m.CampaignName like '%Home%'
        and     m.CampaignName like '%Lapsed%'
        and     m.MessageType = 'EMAIL'
        and     m.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and m.EventDateTime < timestamp '${ads.end_time}'
        group by m.PolicyCode, date_trunc('MONTH', m.EventDateTime)
        having  max(case when m.InteractionType = 'sent' then 1 else 0 end) = 1
        and     (max(case when m.InteractionType in ('open', 'click') then 1 else 0 end) = 0
                 or max(case when m.InteractionType = 'bounce' then 1 else 0 end) = 1)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR3.B1.O1: Renewed inside the window


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr3_b1_o1  (HR3.B1.O1)                                                  */
/*  Description : Renewals / Outcome. Notebook 04, corrected (D-E1.21): renewed after the due date but  */
/*              : inside the window, within due + 7 days online (renewal channel 'e) Online' or a porta */
/*              : l payment) or due + 30 days through the call centre; at the renewal sale date. One ev */
/*              : ent per cycle.                                                                        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyRenewal, PolicySale, RenewalPortalEvent */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr3_b1_o1 as
select  xxhash64('HR3.B1.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR3.B1.O1'                                         as EventTypeId,
        'Renewed inside the window' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(RenewedDate as timestamp)                                     as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Home'
        and     RenewedInWindow = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR3.B1.O3: Lost


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr3_b1_o3  (HR3.B1.O3)                                                  */
/*  Description : Renewals / Outcome. Notebook 04, corrected (D-E1.21, H10): lost = not renewed within  */
/*              : the window (due + 7 online, due + 30 call centre), from the cycle's own result; at du */
/*              : e + 30 days, once that has passed. One event per cycle.                               */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyRenewal, PolicySale, RenewalPortalEvent */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr3_b1_o3 as
select  xxhash64('HR3.B1.O3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR3.B1.O3'                                         as EventTypeId,
        'Lost' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(date_add(RenewalDate, 30) as timestamp)                       as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Home'
        and     Lost = 1
        and     OutcomeFinal = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR5: Renewal processed-  policy continued


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr5  (HR5)                                                              */
/*  Description : Renewals / Spine. Notebook 04 cell 216. A suitability statement on the new policy (04 */
/*              : 's list, mirrored from Motor R5, kept as written).                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyDocumentEvent, PolicyRenewal            */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr5 as
select  xxhash64('HR5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR5'                                               as EventTypeId,
        'Renewal processed-  policy continued' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Home'
        and     d.PolicyTypeGroup = 'Home'
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        and     d.EventDescription in ('Renewal Confirmation - Emailed Document - Suitability Statement',
                                       'Renewal Transfer - Emailed Document - Motor Suitability Statement',
                                       'New Business - Emailed Document - Suitability Statement',
                                       'Renewal Confirmation - Document Transmitted - Suitability Statement',
                                       'Renewal Transfer - Document Transmitted - Motor Suitability Statement',
                                       'New Business - Document Transmitted - Suitability Statement',
                                       'New Business - Document Transmitted - Motor Suitability Statement')
        group by b.PolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR6: Renewal documents dispatched


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr6  (HR6)                                                              */
/*  Description : Renewals / Spine. Notebook 04 cell 217. A certificate or cert issue letter on the new */
/*              :  policy (04's list, mirrored from Motor R6).                                          */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyDocumentEvent, PolicyRenewal            */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr6 as
select  xxhash64('HR6', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR6'                                               as EventTypeId,
        'Renewal documents dispatched' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Home'
        and     d.PolicyTypeGroup = 'Home'
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        and     d.EventDescription in ('Renewal Confirmation - Document Transmitted - RNL Cert Issue Letter',
                                       'Renewal Confirmation - Document Transmitted - Certificate',
                                       'Renewal Transfer - Document Transmitted - RNL Cert Issue Letter',
                                       'Renewal Transfer - Document Transmitted - Certificate',
                                       'New Business - Document Transmitted - NB Cert Issue Letter',
                                       'New Business - Document Transmitted - Certificate',
                                       'Renewal Confirmation - Emailed Document - Certificate',
                                       'Renewal Transfer - Emailed Document - Certificate',
                                       'Renewal Transfer - Printed Document - RNL Cert Issue Letter')
        group by b.PolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR6.F1: The suitability letter barely goes out


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr6_f1  (HR6.F1)                                                        */
/*  Description : Renewals / Friction. Notebook 04 cell 218. The first suitability statement emailed or */
/*              :  transmitted on the new policy (not an offer).                                        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyDocumentEvent, PolicyRenewal            */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr6_f1 as
select  xxhash64('HR6.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR6.F1'                                            as EventTypeId,
        'The suitability letter barely goes out' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Home'
        and     d.PolicyTypeGroup = 'Home'
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        and     d.EventDescription like '%Suitability%'
        and     d.EventDescription not like 'Renewal Offer - %'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        group by b.PolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR4.B2.F1: Chases before the customer submits-  and documents never submitted


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr4_b2_f1  (HR4.B2.F1)                                                  */
/*  Description : Renewals / Friction. Notebook 04 cell 213. Renewal home policies with one of seven do */
/*              : cuments outstanding, latest sale date per policy and month.                           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: DocumentChaseHome                             */
/*              : Reads: ods_v2.DocumentChaseHome                                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr4_b2_f1 as
select  xxhash64('HR4.B2.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR4.B2.F1'                                         as EventTypeId,
        'Chases before the customer submits-  and documents never submitted' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01')) as timestamp)               as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.ods}.DocumentChaseHome
        where   cast(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
        and     PolicyType = 'Renewals'
        and     PolicyTypeGroup = 'Home'
        and     (Comp_DDM_Status = 'O' or Gap_In_Cov_Ltr_Status = 'O' or Val_For_Spec_Item_Status = 'O' or PPS_Num_Status = 'O' or Identification_Status = 'O' or Digital_Journey_Status = 'O' or Finance_Form_Status = 'O')
        group by PolicyCode, date_trunc('MONTH', coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01'))
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR4.B2.4: Chase-  first reminder


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr4_b2_4  (HR4.B2.4)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 234. As HR4.B2.F1 on the DAY 1 campaign, at the f */
/*              : irst chase record per policy and month.                                               */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: DocumentChaseHome                             */
/*              : Reads: ods_v2.DocumentChaseHome                                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr4_b2_4 as
select  xxhash64('HR4.B2.4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR4.B2.4'                                          as EventTypeId,
        'Chase-  first reminder' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(min(CreateDate) as timestamp)                              as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.ods}.DocumentChaseHome
        where   cast(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
        and     PolicyType = 'Renewals'
        and     PolicyTypeGroup = 'Home'
        and     Campaign = 'DAY 1'
        and     (Comp_DDM_Status = 'O' or Gap_In_Cov_Ltr_Status = 'O' or Val_For_Spec_Item_Status = 'O' or PPS_Num_Status = 'O' or Identification_Status = 'O' or Digital_Journey_Status = 'O' or Finance_Form_Status = 'O')
        group by PolicyCode, date_trunc('MONTH', CreateDate)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR4.B2.1: Document request issued


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr4_b2_1  (HR4.B2.1)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 232. A document checklist (SMS / email) saved on  */
/*              : the new policy, first per month. Change: Relay documents only (04 read every ods.Even */
/*              : tStream source here).                                                                 */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyDocumentEvent, PolicyRenewal            */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr4_b2_1 as
select  xxhash64('HR4.B2.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR4.B2.1'                                          as EventTypeId,
        'Document request issued' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.TYPolicyCode                                                  as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Home'
        and     d.PolicyTypeGroup = 'Home'
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        and     d.EventDescription in ('New Business - Saved Document - Document Checklist SMS',
                                       'New Business - Saved Document - Document Checklist Email')
        group by b.TYPolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR4.B2.2: Customer submits documents


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr4_b2_2  (HR4.B2.2)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 233. Document uploads on the new policy, latest p */
/*              : er policy and month.                                                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: DocumentUpload, PolicyRenewal                 */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.DocumentUpload                                  */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr4_b2_2 as
select  xxhash64('HR4.B2.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR4.B2.2'                                          as EventTypeId,
        'Customer submits documents' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(cast(u.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                b.TYPolicyCode                                                  as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.DocumentUpload u     on u.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Home'
        and     cast(u.EventTimestamp as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(u.EventTimestamp as timestamp) < add_months(timestamp '${ads.end_time}', 1)
        group by b.TYPolicyCode, date_trunc('MONTH', cast(u.EventTimestamp as timestamp))
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR4.B2.5: Chase-  escalation


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr4_b2_5  (HR4.B2.5)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 235. A final chase document on the new policy (an */
/*              : y product, as 04), latest per month.                                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyDocumentEvent, PolicyRenewal            */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr4_b2_5 as
select  xxhash64('HR4.B2.5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR4.B2.5'                                          as EventTypeId,
        'Chase-  escalation' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Home'
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < add_months(timestamp '${ads.end_time}', 1)
        and     d.EventDescription like '%Chase%'
        and     d.EventDescription like '%Final%'
        group by b.PolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR4.B2.3: Documents validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr4_b2_3  (HR4.B2.3)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 236. Accepted documents on the new policy in the  */
/*              : window, latest per policy and month.                                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: DocumentWorkflowStatus, PolicyRenewal         */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.DocumentWorkflowStatus                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr4_b2_3 as
select  xxhash64('HR4.B2.3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR4.B2.3'                                          as EventTypeId,
        'Documents validated' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(cast(w.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.DocumentWorkflowStatus w on w.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Home'
        and     cast(w.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(w.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        and     w.IsAccepted = true
        group by b.PolicyCode, date_trunc('MONTH', cast(w.EventTimestamp as timestamp))
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR4.B2.6: Documents received and validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr4_b2_6  (HR4.B2.6)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 238. The same rule as HR4.B2.3 (as today).        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: DocumentWorkflowStatus, PolicyRenewal         */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.DocumentWorkflowStatus                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr4_b2_6 as
select  xxhash64('HR4.B2.6', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR4.B2.6'                                          as EventTypeId,
        'Documents received and validated' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(cast(w.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.DocumentWorkflowStatus w on w.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Home'
        and     cast(w.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(w.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        and     w.IsAccepted = true
        group by b.PolicyCode, date_trunc('MONTH', cast(w.EventTimestamp as timestamp))
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR4.B2.O1: Cancellation-  non-receipt


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr4_b2_o1  (HR4.B2.O1)                                                  */
/*  Description : Renewals / Outcome. Notebook 04 cell 237. An insurer-led cancellation (registration)  */
/*              : email or transmission on the new policy, first per month.                             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyDocumentEvent, PolicyRenewal            */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr4_b2_o1 as
select  xxhash64('HR4.B2.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR4.B2.O1'                                         as EventTypeId,
        'Cancellation-  non-receipt' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Home'
        and     d.PolicyTypeGroup = 'Home'
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        and     d.EventDescription not like 'Renewal Offer - %'
        and     d.EventDescription like '%Insurer Led Cancelation % Reg canx email template%'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        group by b.PolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR6.B1.1: Policy documents dispatched


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr6_b1_1  (HR6.B1.1)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 239. Any document emailed or transmitted on the n */
/*              : ew policy of a retained home renewal (not an offer or a chase). Change: Relay documen */
/*              : ts only (04 read every ods.EventStream source here).                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyDocumentEvent, PolicyRenewal            */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr6_b1_1 as
select  xxhash64('HR6.B1.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR6.B1.1'                                          as EventTypeId,
        'Policy documents dispatched' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Home'
        and     b.PolicyRetNum = 1
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        and     d.EventDescription not like 'Renewal Offer%'
        and     d.EventDescription not like 'Document Chase%'
        group by b.PolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR6.B1.2: Document pack dispatched


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr6_b1_2  (HR6.B1.2)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 240. The same rule as HR6.B1.1 (as today).        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyDocumentEvent, PolicyRenewal            */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr6_b1_2 as
select  xxhash64('HR6.B1.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR6.B1.2'                                          as EventTypeId,
        'Document pack dispatched' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Home'
        and     b.PolicyRetNum = 1
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        and     d.EventDescription not like 'Renewal Offer%'
        and     d.EventDescription not like 'Document Chase%'
        group by b.PolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HR6.B1.5: Replacement requested


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hr6_b1_5  (HR6.B1.5)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 241. A duplicate certificate on the new policy (r */
/*              : eplacement requested).                                                                */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyDocumentEvent, PolicyRenewal            */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hr6_b1_5 as
select  xxhash64('HR6.B1.5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HR6.B1.5'                                          as EventTypeId,
        'Replacement requested' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Home'
        and     d.PolicyTypeGroup = 'Home'
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        and     d.EventDescription like 'Duplicate Certificate%'
        group by b.PolicyCode, date_trunc('MONTH', d.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

In [ ]:
%%pyspark
# Events per family (a family with no events is reported, not failed: check its source and window)
for t in ["adsv2_evt_hr0", "adsv2_evt_hr0_f1", "adsv2_evt_hr0_f2", "adsv2_evt_hr0_f3", "adsv2_evt_hr1a", "adsv2_evt_hr1b", "adsv2_evt_hr1_e1", "adsv2_evt_hr1_e1_f1", "adsv2_evt_hr2a", "adsv2_evt_hr2b", "adsv2_evt_hr2_f1", "adsv2_evt_hr3b_f1", "adsv2_evt_hr3a", "adsv2_evt_hr4a", "adsv2_evt_hr4c", "adsv2_evt_hr3b", "adsv2_evt_hr3_o1", "adsv2_evt_hr4", "adsv2_evt_hr3_b1_1", "adsv2_evt_hr2_b1_1", "adsv2_evt_hr2_b1_f1", "adsv2_evt_hr2_b1_2", "adsv2_evt_hr2_b1_2_f1", "adsv2_evt_hr2_b1_3_f1", "adsv2_evt_hr2_b1_o1", "adsv2_evt_hr2_b1_o2", "adsv2_evt_hr2_b1_o3", "adsv2_evt_hr3_b1_1_f1", "adsv2_evt_hr3_b1_2", "adsv2_evt_hr3_b1_2_f1", "adsv2_evt_hr3_b1_o1", "adsv2_evt_hr3_b1_o3", "adsv2_evt_hr5", "adsv2_evt_hr6", "adsv2_evt_hr6_f1", "adsv2_evt_hr4_b2_f1", "adsv2_evt_hr4_b2_4", "adsv2_evt_hr4_b2_1", "adsv2_evt_hr4_b2_2", "adsv2_evt_hr4_b2_5", "adsv2_evt_hr4_b2_3", "adsv2_evt_hr4_b2_6", "adsv2_evt_hr4_b2_o1", "adsv2_evt_hr6_b1_1", "adsv2_evt_hr6_b1_2", "adsv2_evt_hr6_b1_5"]:
    n = spark.table(f"{stg_schema}.{t}").count()
    print(f"{t:<40} {n:>12,}" + ("   <- no events" if n == 0 else ""))
ads_run_log("14_Home_Renewal", "succeeded")
